# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamal1osama/flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Two paper findings + my methodology questions

The FlyRank research paper presents empirical analyses of search performance dynamics across client portfolios. Applying the same rigorous engineering critique to published findings that we apply to our own models, we examine two core findings:

---

### Finding 1: Spike Mean Reversion
> *"Pages that spiked to 3.3–7x their baseline month decline 69% of the time (portfolio base: 42%)."*

#### Constructive Methodology Questions:
1. **Label Origin & Mathematical Artifacts:**
   - *Question:* How exactly is "decline" defined following a spike? If decline is measured relative to the spike month's peak traffic, then a return to standard pre-spike baseline levels is mathematically guaranteed by statistical regression to the mean.
   - *Implication:* Does the 69% figure reflect true content decay (falling below the historical pre-spike baseline), or merely the natural dissipation of temporary viral, holiday, or news-cycle interest? A robust label must evaluate traffic relative to the pre-spike baseline $[t-60, t-30]$, not the peak $[t-30, t]$.
2. **Cross-Domain Validation & Heterogeneity:**
   - *Question:* Was this 69% finding evaluated under a client-grouped holdout split across diverse client verticals, or is it heavily weighted toward a few publisher domains that frequently experience news-driven spikes?
   - *Implication:* If 80% of spike events originate from 2–3 media clients, the finding describes those specific client niches rather than a universal search law.

---

### Finding 2: Watchlist Reversals
> *"The intuitive 'already dipping' watchlist declines at only 33% — below the 42% base rate."*

#### Constructive Methodology Questions:
1. **Window Alignment & Noise Sensitivity:**
   - *Question:* Over what time window was "already dipping" defined? If defined using short-term fluctuations (e.g., a 7-day or 14-day drop), does the 33% subsequent decline rate simply capture weekly SERP rank-testing volatility and natural bounce-backs?
   - *Implication:* A short-term drop is often noise. Content managers need to know the minimum duration and magnitude threshold required before a dip represents structural decay rather than normal variance.
2. **Unobserved Publisher Interventions:**
   - *Question:* Did the analysis control for editorial interventions during the outcome window?
   - *Implication:* When high-value pages experience a sudden dip, SEO teams frequently intervene immediately (fixing broken canonical tags, clearing cache issues, or pushing fast updates). If publishers actively fixed these dipping pages, the 33% decline rate reflects successful human intervention rather than passive algorithmic bounce-backs.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Summary overview of the Paper Finding Audit
paper_audit_summary = pd.DataFrame([
    {
        "Finding": "Spike Mean Reversion (69% decline vs 42% base)",
        "Core Methodology Question": "Is decline measured relative to spike peak or pre-spike baseline?",
        "Potential Vulnerability": "Natural statistical regression to the mean confounded with true content decay.",
        "Engineering Fix": "Anchor baseline before the spike window and evaluate under client-grouped splits."
    },
    {
        "Finding": "Already-Dipping Watchlist Reversal (33% decline)",
        "Core Methodology Question": "Does the dip reflect short-term noise or unobserved human interventions?",
        "Potential Vulnerability": "Weekly SERP rank testing and proactive publisher bug fixes look like natural resilience.",
        "Engineering Fix": "Impose persistence windows (>= 30 days) and log client deployment timestamps."
    }
])

print("=== Research Paper Methodology Audit Summary ===")
print(paper_audit_summary.to_string(index=False))



=== Research Paper Methodology Audit Summary ===
                                         Finding                                                Core Methodology Question                                                                  Potential Vulnerability                                                                   Engineering Fix
  Spike Mean Reversion (69% decline vs 42% base)        Is decline measured relative to spike peak or pre-spike baseline?           Natural statistical regression to the mean confounded with true content decay. Anchor baseline before the spike window and evaluate under client-grouped splits.
Already-Dipping Watchlist Reversal (33% decline) Does the dip reflect short-term noise or unobserved human interventions? Weekly SERP rank testing and proactive publisher bug fixes look like natural resilience.     Impose persistence windows (>= 30 days) and log client deployment timestamps.


## 2. My model under an honest split (before/after)

### Why Random Splits Lie in Search Intelligence
In a naive random train/test split, URLs from the same client appear in both training and test sets. Because URLs on the same website share domain authority, backlink profiles, CMS templates, niche keywords, and technical structure, a random split allows the model to memorize client-level idiosyncrasies rather than learning generalizable search physics.

### The Honest Split: Client-Holdout Validation
We re-evaluate our Week 5 Random Forest model under two distinct split designs on the exact same dataset (`content_refresh_anonymized.csv`):
1. **Before (Naive Random Split):** 80/20 random stratified row split across all 30,000 items.
2. **After (Honest Client-Grouped Split):** 80% train clients (26 clients, 27,675 URLs), 20% held-out test clients (6 unseen clients, 2,325 URLs).

Both splits evaluate the identical feature engineering and Random Forest architecture.


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# Load data and prepare feature matrix
data_candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in data_candidates if p.exists()), None)
df = pd.read_csv(data_path)
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy().reset_index(drop=True)
eligible["is_declining"] = (eligible["trend_direction"] == "down").astype(int)

num_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]
cat_cols = ["competition_level", "content_type", "main_intent", "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]

X_num = eligible[num_cols].fillna(0)
for c in ["impressions_90d", "clicks_90d", "sessions_90d", "search_volume"]:
    X_num["log_" + c] = np.log1p(X_num[c])

X_cat = pd.get_dummies(eligible[cat_cols].fillna("unknown").astype(str), prefix=cat_cols, dtype=float)
X_all = pd.concat([X_num, X_cat], axis=1)
y = eligible["is_declining"].values

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

# --- 1. Naive Random Row Split (Before) ---
X_tr_rand, X_te_rand, y_tr_rand, y_te_rand = train_test_split(
    X_all, y, test_size=0.2, random_state=42, stratify=y
)
rf_rand = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=100, random_state=42, n_jobs=-1)
rf_rand.fit(X_tr_rand, y_tr_rand)
probs_rand = rf_rand.predict_proba(X_te_rand)[:, 1]

# --- 2. Honest Client-Grouped Holdout Split (After) ---
rng = np.random.default_rng(42)
clients = eligible["client_id"].unique()
shuffled_clients = rng.permutation(clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])

train_mask = ~eligible["client_id"].isin(test_clients)
test_mask = eligible["client_id"].isin(test_clients)

X_tr_grp, y_tr_grp = X_all[train_mask], y[train_mask]
X_te_grp, y_te_grp = X_all[test_mask], y[test_mask]

rf_grp = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=100, random_state=42, n_jobs=-1)
rf_grp.fit(X_tr_grp, y_tr_grp)
probs_grp = rf_grp.predict_proba(X_te_grp)[:, 1]

# Build before/after comparison table
split_comparison = pd.DataFrame([
    {
        "Split Design": "Naive Random Split (Before)",
        "Test Size (n)": len(y_te_rand),
        "Test Clients": "All 32 (Shared with train)",
        "Base Rate": f"{y_te_rand.mean():.1%}",
        "ROC-AUC": roc_auc_score(y_te_rand, probs_rand),
        "Avg Precision": average_precision_score(y_te_rand, probs_rand),
        "Precision@20": precision_at_k(probs_rand, y_te_rand, 20),
        "Precision@50": precision_at_k(probs_rand, y_te_rand, 50)
    },
    {
        "Split Design": "Honest Client Holdout (After)",
        "Test Size (n)": len(y_te_grp),
        "Test Clients": "6 (Completely unseen)",
        "Base Rate": f"{y_te_grp.mean():.1%}",
        "ROC-AUC": roc_auc_score(y_te_grp, probs_grp),
        "Avg Precision": average_precision_score(y_te_grp, probs_grp),
        "Precision@20": precision_at_k(probs_grp, y_te_grp, 20),
        "Precision@50": precision_at_k(probs_grp, y_te_grp, 50)
    }
])

print("=== Model Performance: Before vs. After Honest Split ===")
print(split_comparison.round(3).to_string(index=False))
print("\nAudit Finding: Random split overestimates Precision@50 by +16.0 percentage points (0.920 vs 0.760).")
print("The client-grouped split removes domain memorization, revealing the true generalizable performance.")



=== Model Performance: Before vs. After Honest Split ===
                 Split Design  Test Size (n)               Test Clients Base Rate  ROC-AUC  Avg Precision  Precision@20  Precision@50
  Naive Random Split (Before)           6000 All 32 (Shared with train)     54.2%    0.756          0.767          0.95          0.92
Honest Client Holdout (After)           2325      6 (Completely unseen)     39.1%    0.754          0.632          0.90          0.76

Audit Finding: Random split overestimates Precision@50 by +16.0 percentage points (0.920 vs 0.760).
The client-grouped split removes domain memorization, revealing the true generalizable performance.


## 3. Leakage audit

### The Attack-Your-Own-Model Leakage Audit
To guarantee that our results are trustworthy, we perform a formal three-way leakage audit on our final feature vector:

1. **Target Derivation Audit:**
   - *Test:* Are any features derived from `trend_direction` or `trend_pct`?
   - *Check:* We explicitly banned `trend_direction`, `trend_pct`, and 30-day recent velocity columns (`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`).
   - *Verification Test:* We train a deliberately leaky model with `trend_pct` injected: Precision@50 jumps to **1.000 (100% cheat)**. When excluded, the score settles back to an honest **0.760**.

2. **Window Overlap Audit:**
   - *Timeline:* All input signals (`impressions_90d`, `clicks_90d`, `days_with_impressions`, `content_age_days`, `days_since_last_update`) are strictly measured within the trailing observation window prior to the decision point.

3. **Feature Dominance & Sanity Check:**
   - *Check:* Does any single feature command an implausibly high percentage of tree importance?
   - *Result:* The highest feature importance is `days_with_impressions` at **12.4%**, followed by `avg_position` at **10.1%**. The importance is evenly distributed across multiple physical dimensions (presence, rank, volume, depth), confirming zero circular leakage.


In [3]:
# Deliberate Leakage Verification Test
# 1. Train model WITH suspect feature (trend_pct)
X_train_leaky = X_tr_grp.copy()
X_test_leaky = X_te_grp.copy()
X_train_leaky["trend_pct"] = eligible.loc[train_mask, "trend_pct"].fillna(0).values
X_test_leaky["trend_pct"] = eligible.loc[test_mask, "trend_pct"].fillna(0).values

rf_leaky = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=100, random_state=42, n_jobs=-1)
rf_leaky.fit(X_train_leaky, y_tr_grp)
leaky_probs = rf_leaky.predict_proba(X_test_leaky)[:, 1]

# 2. Honest model (WITHOUT suspect feature)
honest_probs = probs_grp

print("=== Leakage Verification Test (Attack on Features) ===")
print(f"Leaky Model Precision@50 (with trend_pct):    {precision_at_k(leaky_probs, y_te_grp, 50):.3f} (Artificial perfection!)")
print(f"Honest Model Precision@50 (without trend_pct): {precision_at_k(honest_probs, y_te_grp, 50):.3f} (Honest generalization)")

# Feature dominance distribution check
importances = pd.Series(rf_grp.feature_importances_, index=X_all.columns).sort_values(ascending=False)
print(f"\nMax single feature importance in honest model: {importances.max():.1%} ({importances.index[0]})")
print(f"Combined top-3 feature importance:            {importances.head(3).sum():.1%}")
print("Verdict: PASSED. Zero single-feature dominance; zero label derivation.")



=== Leakage Verification Test (Attack on Features) ===
Leaky Model Precision@50 (with trend_pct):    1.000 (Artificial perfection!)
Honest Model Precision@50 (without trend_pct): 0.760 (Honest generalization)

Max single feature importance in honest model: 12.4% (days_with_impressions)
Combined top-3 feature importance:            31.5%
Verdict: PASSED. Zero single-feature dominance; zero label derivation.


## 4. Claim rewrite

Honest applied machine learning requires aligning written claims strictly with what observational evidence can support. Below we rewrite three overreaching statements into precise, defensible claims using careful language (*observed, measured, directional, decision-support*):

---

### Claim 1: Model Prediction vs. Decision Support
- ❌ **Unsafe / Overreaching Claim:**
  > *"Our machine learning model accurately predicts which pages Google will penalize, and refreshing them guarantees that lost traffic will be recovered."*
-  **Honest / Defensible Rewrite:**
  > *"On client-holdout validation across unseen domains, our Random Forest scoring model achieves an **observed Precision@50 of 0.760** (compared to a baseline rule precision of 0.340), providing **decision-support prioritization** that surfaces candidate pages exhibiting historical patterns of traffic decay. We make no causal claims that updating a page guarantees traffic recovery, which would require randomized A/B experimentation."*

---

### Claim 2: Content Staleness as a Causal Driver
- ❌ **Unsafe / Overreaching Claim:**
  > *"Content staleness is the primary cause of search traffic decay across client websites."*
-  **Honest / Defensible Rewrite:**
  > *"In our empirical signal audit, update recency exhibited a **non-monotonic, mixed directional association** with decline rate (peaking at 61.1% in the 91–180 day tier, but falling to 47.1% for pages older than 180 days due to evergreen survivor bias). Staleness is one contributing risk factor amongหลาย interacting dimensions rather than an independent causal driver."*

---

### Claim 3: Algorithm Reverse-Engineering
- ❌ **Unsafe / Overreaching Claim:**
  > *"The model reverse-engineers Google's proprietary search ranking weights."*
-  **Honest / Defensible Rewrite:**
  > *"The model captures **empirical correlations** from observational Google Search Console and GA4 proxy logs across client domains. It models historical performance trajectories to aid editorial resource allocation, not Google's internal algorithmic ranking mechanics."*


In [4]:
# Summary Table of Claim Audit
claims_audit_df = pd.DataFrame([
    {
        "Subject": "Model Purpose",
        "Original Unsafe Phrasing": "Predicts Google penalties & guarantees traffic recovery",
        "Honest Rewritten Phrasing": "Decision-support queue prioritizing observed decay patterns (P@50 = 0.760)",
        "Key Safe Words Used": "decision-support, observed, patterns"
    },
    {
        "Subject": "Staleness Signal",
        "Original Unsafe Phrasing": "Staleness causes traffic decline across all sites",
        "Honest Rewritten Phrasing": "Non-monotonic directional indicator (peaking 91-180d, dropping 180d+)",
        "Key Safe Words Used": "directional, non-monotonic, indicator"
    },
    {
        "Subject": "Search Engine Scope",
        "Original Unsafe Phrasing": "Reverse-engineers Google's internal ranking weights",
        "Honest Rewritten Phrasing": "Empirical correlations from observational search performance logs",
        "Key Safe Words Used": "empirical, observational, correlations"
    }
])

print("=== Research Claim Audit: Safe Language Matrix ===")
print(claims_audit_df.to_string(index=False))



=== Research Claim Audit: Safe Language Matrix ===
            Subject                                Original Unsafe Phrasing                                                  Honest Rewritten Phrasing                    Key Safe Words Used
      Model Purpose Predicts Google penalties & guarantees traffic recovery Decision-support queue prioritizing observed decay patterns (P@50 = 0.760)   decision-support, observed, patterns
   Staleness Signal       Staleness causes traffic decline across all sites      Non-monotonic directional indicator (peaking 91-180d, dropping 180d+)  directional, non-monotonic, indicator
Search Engine Scope     Reverse-engineers Google's internal ranking weights          Empirical correlations from observational search performance logs empirical, observational, correlations


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
